# Running LightCurveLynx Through Rubin's Metric Analysis Framework

LightCurveLynx has been integrated with [rubin_sim](https://rubin-sim.lsst.io/)'s Metric Analysis Framework (MAF) to allow users to combine the two packages to conduct various studies. In order to use LightCurveLynx and MAF together, you will need to install both packages separately. For installation instructions, see the [rubin_sim documentation](https://rubin-sim.lsst.io/).

We support two different approaches for using LightCurveLynx simulations in MAF:

1. we allow MAF to read and analyze the results of a LightCurveLynx simulation. 
2. we allow MAF to call out directly to LightCurveLynx to perform a dynamic simulation.

The first approach will generally be more efficient, because it can take advantage of LightCurveLynx's batching and vectorization. However, the second approach may work better when the analysis is part of a broader optimization loop. 

In this notebook, we will show examples of both approaches.

## Setup MAF

We start by importing the libraries and data we need to run MAF. This includes the `rubin_sim.maf` subpackage and the current baseline OpSim.

In [ ]:
from rubin_sim import maf
import tempfile

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

opsim_fname = get_baseline()
print(f"Using the {opsim_fname} file for the baseline opsim data")

data_dir = tempfile.mkdtemp()
print(f"Using the {data_dir} directory for output of this notebook")

## Setup LightCurveLynx

Next we import the libraries we need to run LightCurveLynx and create a simple model for our simulation. We load the baseline OpSim from the previous section and use it to define our sampling region for RA and Dec. Note that, due to the size of the OpSim, it may take a few minutes to load it and construct a MOC.

In [ ]:
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.math_nodes.ra_dec_sampler import ApproximateMOCSampler
from lightcurvelynx.models.basic_models import SinWaveModel
from lightcurvelynx.obstable.opsim import OpSim

# Load the OpSim data.
ops_data = OpSim.from_db(opsim_fname)

# Create a sampler for the RA/Dec coordinates based on the OpSim coverage.
ra_dec_sampler = ApproximateMOCSampler.from_obstable(ops_data, depth=8)

model = SinWaveModel(
    brightness=2000.0,
    amplitude=200.0,
    frequency=NumpyRandomFunc("uniform", low=0.01, high=0.1),
    t0=NumpyRandomFunc("uniform", low=0.0, high=10.0),
    ra=ra_dec_sampler.ra,
    dec=ra_dec_sampler.dec,
    node_label="sin_wave_model",
)

## MAF on LightCurveLynx Output

Our first approach is to run a full LightCurveLynx simulation, write the results out, and have MAF analyze those files. We start by running the simulation and saving the results to a file.

In [ ]:
from lightcurvelynx.simulate import simulate_lightcurves
from lightcurvelynx.survey_info import SurveyInfo
from lightcurvelynx.utils.post_process_results import results_augment_lightcurves

# Simulate light curves using the specified model and survey information.
lightcurves = simulate_lightcurves(
    model,  # The model to simulate (including effects).
    1000,  # The number of light curves to simulate.
    SurveyInfo(ops_data),  # The survey information.
)

# Augment the simulated light curves with additional information if needed.
lightcurves = results_augment_lightcurves(lightcurves)

results_file_name = f"{data_dir}/results.parquet"
lightcurves.to_parquet(results_file_name)
print(f"Saved the results to {results_file_name}")

Next we set up the MAF Slicer and Metrics that we want to use. For a good introduction to how to use MAF, see their [tutorial notebooks](https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/tutorial). We use the `LynxResultsSlicer`, which reads the points from a LightCurveLynx output file, and the `LCLObsCountMetric`, which counts the number of observations per light curve.

In [ ]:
from rubin_sim.maf.metrics.base_lynx_metrics import LynxObsCountMetric
from rubin_sim.maf.metric_bundles.metric_bundle import MetricBundle
from rubin_sim.maf.metric_bundles.metric_bundle_group import MetricBundleGroup
from rubin_sim.maf.slicers.lynx_slicer import LynxResultsSlicer

# Create the metric bundle and the metric bundle group
bundle = MetricBundle(
    metric=LynxObsCountMetric(from_results=True),
    slicer=LynxResultsSlicer(),
    constraint="",
)

bundle_group = MetricBundleGroup(
    [bundle],
    results_file_name,
    out_dir=data_dir,
)

# Run the metric bundle group
bundle_group.run_all()

We can plot a histogram of the raw counts.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(bundle.metric_values)
plt.xlabel("Metric Value")
plt.ylabel("Frequency")
plt.title("Histogram of Metric Values")
plt.show()

## LightCurveLynx through MAF

Our second approach is to dynamically run a LightCurveLynx simulation during the MAF run. We use the `LynxSamplerSlicer`, which dynamically simulates the observations with LightCurveLynx, and the `LCLObsCountMetric`, which counts the number of observations per light curve.

In [ ]:
from rubin_sim.maf.slicers.lynx_slicer import LynxSamplerSlicer

bundle2 = maf.MetricBundle(
    LynxObsCountMetric(from_results=False),
    LynxSamplerSlicer(1000, model),  # 1000 samples from model.
    constraint="",
)
bundle_group2 = maf.MetricBundleGroup([bundle2], opsim_fname, out_dir=data_dir)
bundle_group2.run_all()

We can then use MAF's built-in functions to summarize the results.

In [ ]:
bundle_group2.plot_all(closefigs=False)

## Clean up

As a final step, we need to clean up the temporary directory we created for the files.

In [ ]:
import os
import shutil

if os.path.exists(data_dir):
    shutil.rmtree(data_dir)